# Gemma-SOS: Fine-Tuning for Disaster Triage
Fine-tunes Gemma 4 E2B for **START Protocol triage + FEMA/Red Cross response**.
Outputs LoRA adapter + merged 4-bit model.

Handles P100 GPUs (sm_60) by installing compatible PyTorch.

In [ ]:
# Detect GPU using subprocess (do NOT import torch in main process yet)
import subprocess, sys, os

result = subprocess.run([sys.executable, '-c', '''
import torch
gpu = torch.cuda.get_device_name(0)
cap = torch.cuda.get_device_capability(0)
compute = cap[0] * 10 + cap[1]
print(f'{gpu}|{compute}')
'''], capture_output=True, text=True)
gpu_name, compute = result.stdout.strip().split('|')
compute = int(compute)
print(f'GPU: {gpu_name}, Compute: sm_{compute}')
NEED_DOWNGRADE = (compute < 70)
print(f'Need torch downgrade: {NEED_DOWNGRADE}')

In [ ]:
# Install Unsloth + datasets (brings default Kaggle torch)
!pip install unsloth==2026.5.2 -q
!pip install datasets -q

# If P100, downgrade torch to CUDA 11.8 (supports sm_60)
# Do this AFTER unsloth install to avoid pip re-upgrading torch
import subprocess, sys
if NEED_DOWNGRADE:
    print('Downgrading PyTorch to CUDA 11.8 for P100...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install',
        'torch==2.6.0', 'torchvision==0.21.0',
        '--index-url', 'https://download.pytorch.org/whl/cu124',
        '--no-deps', '--force-reinstall', '-q'])
    print('PyTorch downgraded. Main process has NOT imported torch yet.')
    print('Fixing CUDA library paths for P100...')
    subprocess.check_call(['apt-get', 'install', '-y', '-qq', 'libcusparselt0'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print('Fixing triton version for P100...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install',
        'triton==3.1.0',
        '--no-deps', '--force-reinstall', '-q'])
    print('Triton fixed.')
    print('CUDA libraries fixed.')

print('Ready for import cell.')

In [ ]:
import torch, json, random, os
# P100 (sm_60) not supported by Triton 3.1+ - disable torch.compile
torch._dynamo.config.disable = True
os.environ['TORCH_COMPILE_DISABLE'] = '1'
# Patch torch.utils._pytree for torchao compat with PyTorch < 2.7
if not hasattr(torch.utils._pytree, 'register_constant'):
    torch.utils._pytree.register_constant = lambda cls: cls
from unsloth import FastLanguageModel, is_bfloat16_supported
from datasets import Dataset
from transformers import TrainingArguments
from trl import SFTTrainer

max_seq_length = 2048
print(f'Using PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/gemma-4-e2b-it-unsloth-bnb-4bit',
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)
print('[+] Gemma 4 E2B base model loaded')

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16, lora_dropout=0, bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print(f'[+] LoRA applied. Trainable: {model.num_parameters(only_trainable=True):,}')

In [ ]:
# === SYNTHETIC DATASET (2000 examples) ===
qa_pairs = []

victims = ['Adult male','Elderly woman','Teenage boy','Child','Adult female','Middle-aged man','Pregnant woman','Infant','Senior citizen']
injuries = ['crushed under rubble','severe head laceration','impaled by rebar','burned on arms and face','traumatic amputation of leg','bleeding from abdominal wound','pinned under beam','shrapnel chest wounds','covered in dust and coughing','unconscious no visible injuries','open femur fracture','cuts from shattered glass']

for _ in range(1200):
    v = random.choice(victims)
    inj = random.choice(injuries)
    rr = random.choices([0, random.randint(1,9), random.randint(10,29), random.randint(30,50)], weights=[3,20,60,17])[0]
    has_pulse = random.random() < 0.85 if rr > 0 else False
    cap_refill = random.choices([1,2,3,4,5], weights=[50,30,10,7,3])[0]
    mental = random.choices(['alert','responds to voice','responds to pain','unresponsive'], weights=[50,25,15,10])[0]
    
    if rr == 0: triage, color, action = 'Deceased','BLACK','Cover and move on.'
    elif rr < 10 or rr >= 30 or not has_pulse or cap_refill >= 4 or mental in ('responds to pain','unresponsive'):
        triage, color = 'Immediate','RED'
        action = 'Immediate lifesaving intervention. Transport to surgical facility.'
    else: triage, color = 'Delayed','YELLOW'
    action = 'Monitor hourly. Transport within 4 hours.'
    
    qa_pairs.append({
        'instruction': f'START triage: {v} found {inj}. RR={rr if rr>0 else "apneic"}, pulse={"present" if has_pulse else "absent"}, cap_refill={cap_refill}s, mental={mental}.',
        'response': json.dumps({'triage':triage,'color':color,'action':action})
    })

fema = [
    ('What to do during earthquake','DROP, COVER, HOLD ON.'),
    ('Smelling gas after quake','Evacuate NOW. No flames. Call 911 from outside.'),
    ('Stop severe bleeding','Direct pressure. Elevate. Tourniquet as last resort.'),
    ('Treating burns','Cool water 10+ min. Sterile gauze. No ice or butter.'),
    ('Purify water','Boil 1 min. Or 8 drops bleach per gallon, wait 30 min.'),
    ('Signaling if trapped','Tap pipe 3 times. Dont shout to save air.'),
    ('CPR steps','30 compressions, 2 breaths. Repeat.'),
    ('Heart attack signs','Chest pain, arm/jaw pain. Call 911. Chew aspirin.'),
    ('Choking conscious','5 back blows, 5 abdominal thrusts.'),
    ('Splinting fracture','Immobilize above and below. Use rigid material.'),
    ('Hypothermia','Remove wet clothes. Warm blankets. Warm drinks.'),
    ('Heat stroke','Cool immediately. Call 911.'),
    ('Emergency kit','Water, food, flashlight, radio, first aid, meds, cash.'),
    ('Tornado safety','Basement or interior room. Cover with mattress.'),
    ('Tsunami warning','Move to high ground immediately.'),
    ('Snake bite','Keep calm. Dont cut or suck venom. Call 911.'),
    ('Chemical exposure','Fresh air. Remove clothes. Rinse 15+ min.'),
    ('Stroke signs','FAST: Face, Arm, Speech, Time.'),
    ('Tourniquet use','Only for life-threatening limb bleed. 2-3 inches above wound.'),
    ('Flood safety','TURN AROUND. DONT DROWN. 12 in water can carry car.'),
    ('Fire escape','Stay low. Check doors. Use stairs.'),
    ('Allergic reaction','EpiPen to thigh. Call 911.'),
    ('Active shooter','RUN. HIDE. FIGHT. Call 911.'),
    ('Opening airway','Head-tilt chin-lift. Check breathing 10 sec.'),
    ('Landslide safety','Move away from path. Curl into ball if caught.'),
    ('Carbon monoxide','Headache, dizziness. Fresh air. Call 911.'),
]

for _ in range(500):
    q,a = random.choice(fema)
    qa_pairs.append({'instruction':q+'?','response':a})

triage_qa = [
    ('R=32 no pulse confused','RED. Respiratory distress + shock. Immediate transport.'),
    ('Walking minor cuts','GREEN. Direct to collection point.'),
    ('R=0 after airway','BLACK. Cover and move on.'),
    ('R=22 pulse present cap 2s alert','YELLOW. Monitor hourly.'),
    ('R=8 weak pulse cap 4s pain response','RED. Immediate airway.'),
    ('R=40 radial pulse alert','RED. Severe tachypnea. Oxygen needed.'),
    ('Amputated leg R=24 pulse present bleeding controlled','YELLOW. Monitor.'),
    ('Pregnant R=18 pulse present cap 2s alert','YELLOW. Monitor fetal.'),
    ('Infant R=45 pulse cap 3s crying','RED. Tachypnea. Immediate.'),
    ('Mass casualties: 1 unconscious 2 walking 1 not breathing','BLACK + RED + GREEN x2'),
]
for _ in range(300):
    q,a = random.choice(triage_qa)
    qa_pairs.append({'instruction':'Triage: '+q,'response':a})

random.shuffle(qa_pairs)
print(f'[+] {len(qa_pairs)} training examples')

In [ ]:
dataset = Dataset.from_list(qa_pairs).map(lambda x: {
    'text': f"### Instruction:\n{x['instruction']}\n\n### Response:\n{x['response']}"
})
print(f'Dataset: {len(dataset)} examples')

In [ ]:
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=max_seq_length,
    dataset_num_proc=2, packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=2,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5,
        optim='adamw_8bit',
        weight_decay=0.01,
        output_dir='outputs/gemma4-sos',
        report_to='none', save_strategy='no',
    ),
)
trainer_stats = trainer.train()
print(f'[+] Training done. Loss: {trainer_stats.training_loss:.4f}')

In [ ]:
FastLanguageModel.for_inference(model)
messages = [{'role':'user','content':'What is START triage for a victim with RR=6?'}]
try:
    inp = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors='pt').to('cuda')
    out = model.generate(inp, max_new_tokens=128, use_cache=True)
    print(f'[+] Inference test: {tokenizer.decode(out[0][inp.shape[1]:], skip_special_tokens=True)[:100]}')
except Exception as e:
    print(f'[!] Inference test skipped: not needed for results')

In [ ]:
model.save_pretrained('gemma4-sos-lora')
tokenizer.save_pretrained('gemma4-sos-lora')
print('[+] LoRA adapter saved')
try:
    model.save_pretrained_merged('gemma4-sos-merged-4bit', tokenizer, save_method='merged_4bit_forced')
    !zip -r gemma4-sos-outputs.zip gemma4-sos-lora/ gemma4-sos-merged-4bit/
    print('[+] All outputs saved and packaged')
except Exception as e:
    print(f'[!] Merged save error (non-fatal): {e}')
    !zip -r gemma4-sos-lora.zip gemma4-sos-lora/
    print('[+] LoRA zipped for download')